# Anomaly Comparison: LightGBM vs Imredi
## Overview

This notebook supplements `LightGBM_Final.ipynb` by benchmarking the detected anomalies against an external reference set provided by Imredi — the company that supplied the original dataset. Imredi independently applied their own anomaly detection pipeline to the same August 2025 test period, producing 1,194 flagged events across 494 products.

The comparison is conducted using the same statistical evaluation framework defined in `LightGBM_Final.ipynb`. All four LightGBM configurations are evaluated at z-threshold = −4.5 alongside the Imredi baseline, enabling a direct, metric-consistent assessment of signal quality, product coverage, and operational characteristics.

In [1]:
import pandas as pd
import numpy as np
import os

FULL_DATA_PATH = "data_v1.csv"
TEST_START = "2025-08-01"
ANOMALY_DIR = "LightGBM_final_test_anomalies_z45"
IMREDI_SAMPLE = "target_sample.csv"

ANOMALY_FILES = [
    "4_huber_strong_z45.csv",
    "3_rmse_strong_z45.csv",
    "2_huber_conservative_z45.csv",
    "1_rmse_conservative_z45.csv"
]

MODEL_NAMES = [
    "Original Imredi results",
    "Huber Strong z45",
    "RMSE Strong z45",
    "Huber Conservative z45",
    "RMSE Conservative z45"
]

In [2]:
def evaluate_anomalies(anomaly_df, full_df, test_start="2025-08-01"):
    pre_test = full_df[full_df['date'] < test_start].copy()
    first_pos = (pre_test[pre_test['sales'] > 0]
                 .groupby('product')['date'].min()
                 .reset_index(name='first_positive_date'))
    pre_test = pre_test.merge(first_pos, on='product', how='left')
    pre_test = pre_test[pre_test['date'] >= pre_test['first_positive_date']].copy()

    stats_full = pre_test.groupby('product').agg(
        mean_sales_hist   = ('sales', 'mean'),
        std_sales_hist    = ('sales', 'std'),
        median_sales_hist = ('sales', 'median'),
    ).reset_index()

    stats_july = (pre_test[pre_test['date'].dt.month == 7]
                  .groupby('product').agg(
                      mean_sales_july = ('sales', 'mean'),
                      std_sales_july  = ('sales', 'std'),
                  ).reset_index())

    anom = anomaly_df.copy()
    anom['date']    = pd.to_datetime(anom['date'])
    anom['product'] = anom['product'].astype(str)
    anom['hour']    = anom['date'].dt.hour
    anom['weekday'] = anom['date'].dt.dayofweek

    anom = anom.merge(stats_full, on='product', how='left')
    anom = anom.merge(stats_july, on='product', how='left')

    for col in ['std_sales_hist', 'std_sales_july', 'mean_sales_hist', 'mean_sales_july']:
        anom[col] = anom[col].replace(0, np.nan).fillna(1)

    anom['z_hist'] = (anom['sales'] - anom['mean_sales_hist']) / anom['std_sales_hist']
    anom['z_july'] = (anom['sales'] - anom['mean_sales_july']) / anom['std_sales_july']
    anom['dev_pct_hist'] = (anom['sales'] - anom['mean_sales_hist']) / anom['mean_sales_hist'] * 100
    anom['dev_pct_july'] = (anom['sales'] - anom['mean_sales_july']) / anom['mean_sales_july'] * 100

    stock_prev = full_df[['date', 'product', 'stocks']].rename(columns={'stocks': 'stocks_prev', 'date': 'prev_date'})
    anom['prev_date'] = anom['date'] - pd.Timedelta(hours=1)
    anom = anom.merge(stock_prev, on=['prev_date', 'product'], how='left')

    drop_mask      = anom['sales'] < anom['mean_sales_hist']
    zero_mask      = anom['sales'] == 0
    deep_drop_mask = anom['dev_pct_hist'] < -50
    stock_ok_mask  = (anom['stocks_prev'] > 0) & (anom['stocks'] > 0)
    n = len(anom)

    return {
        'Anomalies found': n,
        'Unique products': anom['product'].nunique(),
        'Unique days': anom['date'].dt.date.nunique(),
        'Anomalies/day (avg)': round(n / max(anom['date'].dt.date.nunique(), 1), 1),

        'Mean deviation from norm (%)': round(anom.loc[drop_mask, 'dev_pct_hist'].mean(), 1),
        'Median deviation from norm (%)': round(anom.loc[drop_mask, 'dev_pct_hist'].median(), 1),
        'Mean deviation from July (%)': round(anom.loc[drop_mask, 'dev_pct_july'].mean(), 1),

        'Mean |z_hist|': round(anom['z_hist'].abs().mean(), 3),
        'Mean |z_july|': round(anom['z_july'].abs().mean(), 3),
        '% |z_hist| > 2': round((anom['z_hist'].abs() > 2).mean() * 100, 1),
        '% |z_hist| > 3': round((anom['z_hist'].abs() > 3).mean() * 100, 1),
        '% |z_july| > 2': round((anom['z_july'].abs() > 2).mean() * 100, 1),
        '% |z_july| > 3': round((anom['z_july'].abs() > 3).mean() * 100, 1),

        '% zero sales': round(zero_mask.mean() * 100, 1),
        '% drops > 50% below norm': round(deep_drop_mask.mean() * 100, 1),
        '% anomalies with stock': round(stock_ok_mask.mean() * 100, 1),

        'Zero stock at anomaly time (count)': int((anom['stocks'] == 0).sum()),
        'Zero stock previous hour (count)': int((anom['stocks_prev'] == 0).sum()),

        'Peak hour (mode)': int(anom['hour'].mode().iloc[0]) if n > 0 else None,
        '% working hours (7–22)': round(((anom['hour'] >= 7) & (anom['hour'] <= 22)).mean() * 100, 1),
        '% weekdays': round((anom['weekday'] < 5).mean() * 100, 1),
    }



In [3]:
df_full = pd.read_csv(FULL_DATA_PATH)
df_full['date'] = pd.to_datetime(df_full['date'])
df_full['product'] = df_full['product'].astype(str)

In [4]:
results = {}

anom_df = pd.read_csv(IMREDI_SAMPLE)
metrics = evaluate_anomalies(anom_df, df_full, TEST_START)
results["Original target_sample"] = metrics

for fname, mname in zip(ANOMALY_FILES, MODEL_NAMES[1:]):
    fpath = os.path.join(ANOMALY_DIR, fname)
    anom_df = pd.read_csv(fpath)
    metrics = evaluate_anomalies(anom_df, df_full, TEST_START)
    results[mname] = metrics

comparison_df = pd.DataFrame(results)

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 500)
pd.set_option('display.precision', 2)


comparison_df

,Original target_sample,Huber Strong z45,RMSE Strong z45,Huber Conservative z45,RMSE Conservative z45
Anomalies found,1194.00,214.00,450.00,150.00,449.00
Unique products,494.00,40.00,51.00,36.00,71.00
Unique days,24.00,30.00,30.00,29.00,30.00
Anomalies/day (avg),49.80,7.10,15.00,5.20,15.00
Mean deviation from norm (%),-99.80,-112.20,-116.80,-113.30,-112.40
Median deviation from norm (%),-100.00,-100.00,-100.00,-100.00,-100.00
Mean deviation from July (%),-99.60,-111.60,-116.50,-112.50,-114.40
Mean |z_hist|,0.57,2.93,4.15,1.92,3.83
Mean |z_july|,0.53,2.16,3.06,1.56,2.78
% |z_hist| > 2,5.70,38.80,54.40,24.00,55.90


## Conclusions

The comparison demonstrates a clear qualitative advantage of the LightGBM-based pipeline over the Imredi reference across all core diagnostic dimensions.

**Statistical signal strength.** The mean historical z-score of detected anomalies ranges from 1.92 to 4.15 across LightGBM configurations, compared to 0.57 for Imredi. The proportion of anomalies exceeding |z_hist| > 2 reaches 54–56% in the best-performing configurations, versus 5.7% in the Imredi set. This indicates that the LightGBM pipeline flags events that are statistically extreme relative to each product's historical baseline, while the majority of Imredi anomalies fall within normal variation.

**Stock filter compliance.** All LightGBM anomalies satisfy the mandatory domain constraint — stock confirmed positive in both the current and preceding hour. The Imredi set contains 40 events with zero stock at anomaly time and 34 with zero stock in the preceding hour, meaning a non-trivial share of their flagged events are attributable to out-of-stock conditions rather than shelf availability failures. This is a critical distinction for the target use case.

**Alert volume and operational utility.** LightGBM produces between 150 and 450 anomalies across configurations at z = −4.5, compared to 1,194 for Imredi. The higher volume of the Imredi set, combined with its weak statistical signal, suggests a lower-precision detection regime that would generate a substantially higher operational load without a corresponding increase in actionable alerts.

**Recommended configuration.** RMSE Strong at z = −4.5 offers the best overall balance: the highest mean |z_hist| (4.15), broad product coverage (51 unique SKUs), full stock filter compliance, and 30-day temporal spread. It is the configuration recommended for production deployment, as established in `LightGBM_Final.ipynb`.

In summary, the LightGBM pipeline consistently identifies anomalies with stronger statistical grounding, stricter domain validity, and a more operationally tractable alert volume than the external reference. The results support the methodological conclusion that residual-based detection with learned product-level expectations outperforms threshold-based approaches on heterogeneous retail catalogues.